# 1. Thiết lập lớp huấn luyện và đánh giá

In [1]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
    matthews_corrcoef, cohen_kappa_score, confusion_matrix, classification_report
)
import numpy as np
import pandas as pd

class ClassificationMetrics:
    def compute(self, y_true, y_pred):
        metrics = {}

        # Overall metrics
        metrics["accuracy"] = accuracy_score(y_true, y_pred)
        metrics["balanced_accuracy"] = balanced_accuracy_score(y_true, y_pred)
        metrics["precision_macro"] = precision_score(y_true, y_pred, average='macro', zero_division=0)
        metrics["precision_weighted"] = precision_score(y_true, y_pred, average='weighted', zero_division=0)
        metrics["recall_macro"] = recall_score(y_true, y_pred, average='macro', zero_division=0)
        metrics["recall_weighted"] = recall_score(y_true, y_pred, average='weighted', zero_division=0)
        metrics["f1_macro"] = f1_score(y_true, y_pred, average='macro', zero_division=0)
        metrics["f1_weighted"] = f1_score(y_true, y_pred, average='weighted', zero_division=0)

        # Overall G-Mean (macro)
        recalls = recall_score(y_true, y_pred, average=None, zero_division=0)
        metrics["gmean"] = np.prod(recalls) ** (1 / len(recalls))

        metrics["mcc"] = matthews_corrcoef(y_true, y_pred)
        metrics["kappa"] = cohen_kappa_score(y_true, y_pred)

        # Confusion matrix
        cm = confusion_matrix(y_true, y_pred)
        metrics["confusion_matrix"] = cm

        # Per-class metrics
        report = classification_report(y_true, y_pred, output_dict=True, zero_division=0)

        # ---- Compute per-class G-Mean ----
        per_class_gmean = {}
        n_classes = cm.shape[0]
        for i in range(n_classes):
            TP = cm[i, i]
            FN = cm[i, :].sum() - TP
            FP = cm[:, i].sum() - TP
            TN = cm.sum() - (TP + FP + FN)

            TPR = TP / (TP + FN) if (TP + FN) > 0 else 0
            TNR = TN / (TN + FP) if (TN + FP) > 0 else 0
            gmean_i = np.sqrt(TPR * TNR)

            per_class_gmean[str(i)] = gmean_i

        # Build per_class output
        metrics["per_class"] = {
            label: {
                "precision": report[label]["precision"],
                "recall": report[label]["recall"],
                "f1-score": report[label]["f1-score"],
                "support": report[label]["support"],
                "gmean": per_class_gmean[label]
            }
            for label in report if label.isdigit()
        }

        metrics["per_class_gmean"] = per_class_gmean

        return metrics


In [2]:
import time

def run_phase_timer(phase_name, fn, *args, **kwargs):
    print(f"\n[PHASE START] {phase_name}")
    start = time.perf_counter()

    output = fn(*args, **kwargs)

    elapsed = time.perf_counter() - start
    print(f"[PHASE END]   {phase_name} | Time: {elapsed:.2f}s")
    return output, elapsed


In [3]:
import pandas as pd
import lightgbm as lgb

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score
from sklearn.model_selection import ParameterGrid


class LGBMTrainer:
    def __init__(self, train_path):
        """
        train_path: đường dẫn file train CSV.
        """
        self.train_path = train_path
        self.train_df = None
        self.X_train = None
        self.y_train = None

        # LightGBM không bắt buộc scale
        self.scaler = StandardScaler()
        self.model = None

        self.best_params = None
        self.best_score = None

    # -----------------------------------------------------
    def load_and_prepare_train(self):
        """Đọc dữ liệu train và xử lý cột"""
        self.train_df = pd.read_csv(self.train_path)

        X = self.train_df.drop(["label_3", "course_id", "user_id"], axis=1)
        y = self.train_df["label_3"]

        # Có thể bỏ scaler nếu muốn
        self.X_train = self.scaler.fit_transform(X)
        self.y_train = y

    # -----------------------------------------------------
    def _prepare_val_or_test(self, df):
        """Chuẩn hóa file val/test để phù hợp train"""
        X_val = df.drop(["label_3", "course_id", "user_id"], axis=1)

        train_cols = self.train_df.drop(
            ["label_3", "course_id", "user_id"], axis=1
        ).columns

        missing_cols = train_cols.difference(X_val.columns)
        for col in missing_cols:
            X_val[col] = self.train_df[col].median()

        X_val = X_val[train_cols]
        X_val = self.scaler.transform(X_val)

        return X_val

    # -----------------------------------------------------
    def train_with_gridsearch(self, val_path, param_grid):
        """
        GridSearch LightGBM sử dụng F1-macro
        """
        print("\n Running Grid Search (LightGBM, F1-macro)...\n")

        val_df = pd.read_csv(val_path)
        y_val = val_df["label_3"]
        X_val = self._prepare_val_or_test(val_df)

        best_f1 = -1
        best_params = None
        best_model = None

        for params in ParameterGrid(param_grid):
            print(f"→ Testing params: {params}")

            model = lgb.LGBMClassifier(
                **params,
                objective="multiclass",
                random_state=42,
                n_jobs=-1,
                class_weight="balanced",  # nếu label_3 lệch
                verbosity=-1
            )


            model.fit(self.X_train, self.y_train)
            y_pred = model.predict(X_val)

            f1 = f1_score(y_val, y_pred, average="macro")
            print(f"   F1-macro = {f1:.4f}")

            if f1 > best_f1:
                best_f1 = f1
                best_params = params
                best_model = model

        self.model = best_model
        self.best_params = best_params
        self.best_score = best_f1

        print("\n Best GridSearch Result:")
        print("Best Params:", best_params)
        print(f"Best F1-macro: {best_f1:.4f}")

        return best_model, best_params, best_f1

    # -----------------------------------------------------
    def evaluate_test(self, test_path, metrics):
        """Đánh giá model trên test set"""
        print(f"\n=== Evaluate Test Set: {test_path} ===\n")

        test_df = pd.read_csv(test_path)
        X_test = self._prepare_val_or_test(test_df)
        y_test = test_df["label_3"]

        y_pred = self.model.predict(X_test)
        results = metrics.compute(y_test, y_pred)

        print("Accuracy:", results["accuracy"])
        print("Macro F1:", results["f1_macro"])
        print("\nConfusion Matrix:")
        print(results["confusion_matrix"])
        print("\nPer-class Metrics:")
        print(results["per_class"])

        return results


# 2 Thực nghiệm

## 2.1 Thực nghiệm Light GNB trên tập Raw

In [4]:
import json
import os
trainer = LGBMTrainer("/kaggle/input/mooccubex-data-cleaned/split_data/data_median_im_3/train.csv")
trainer.load_and_prepare_train()


param_grid = {
    "n_estimators": [200],
    "max_depth": [10],
    "num_leaves": [31, 63, 127],
    "min_child_samples": [20]
}

best_model, best_params, best_f1 = trainer.train_with_gridsearch(
    val_path="/kaggle/input/mooccubex-data-cleaned/split_data/data_median_im_3/val.csv",
    param_grid=param_grid
)

output_dir = "/kaggle/working/raw"
os.makedirs(output_dir, exist_ok=True)   # tạo thư mục nếu chưa có

# Evaluate nhiều test set
for i in range(1, 5):
    results = trainer.evaluate_test(
        f"/kaggle/input/mooccubex-data-cleaned/split_data/data_median_im_3/test_{i}.csv",
        metrics=ClassificationMetrics()
    )

    # Convert numpy types -> native Python
    if isinstance(results.get("confusion_matrix"), np.ndarray):
        results["confusion_matrix"] = results["confusion_matrix"].tolist()

    save_path = f"{output_dir}/phase_{i}_results.json"

    try:
        with open(save_path, "w") as f:
            json.dump(results, f, indent=4)
        print(f"Saved: {save_path}")
    except Exception as e:
        print(f"Error saving phase {i}:", e)


 Running Grid Search (LightGBM, F1-macro)...

→ Testing params: {'max_depth': 10, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 31}
   F1-macro = 0.6536
→ Testing params: {'max_depth': 10, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 63}
   F1-macro = 0.7295
→ Testing params: {'max_depth': 10, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 127}
   F1-macro = 0.7584

 Best GridSearch Result:
Best Params: {'max_depth': 10, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 127}
Best F1-macro: 0.7584

=== Evaluate Test Set: /kaggle/input/mooccubex-data-cleaned/split_data/data_median_im_3/test_1.csv ===

Accuracy: 0.9969456191144016
Macro F1: 0.5154974288692938

Confusion Matrix:
[[    28     19    176]
 [     1    129    475]
 [     0     39 231586]]

Per-class Metrics:
{'0': {'precision': 0.9655172413793104, 'recall': 0.12556053811659193, 'f1-score': 0.22222222222222224, 'support': 223, 'gmean': 0.3543444615672554}, '1': {'precision': 0

## 2.2 Thực nghiệm LightGNB trên tập data_median_sasmote

In [5]:
import json
import os
trainer = LGBMTrainer("/kaggle/input/mooccubex-data-cleaned/split_data/data_median_sasmote/train_median_sasmote.csv")
trainer.load_and_prepare_train()

param_grid = {
    "n_estimators": [200],
    "max_depth": [10],
    "num_leaves": [31, 63, 127],
    "min_child_samples": [20]
}

best_model, best_params, best_f1 = trainer.train_with_gridsearch(
    val_path="/kaggle/input/mooccubex-data-cleaned/split_data/data_median_sasmote/val.csv",
    param_grid=param_grid
)

output_dir = "/kaggle/working/median_sasmote"
os.makedirs(output_dir, exist_ok=True)   # tạo thư mục nếu chưa có

# Evaluate nhiều test set
for i in range(1, 5):
    results = trainer.evaluate_test(
        f"/kaggle/input/mooccubex-data-cleaned/split_data/data_median_sasmote/test_{i}.csv",
        metrics=ClassificationMetrics()
    )

    # Convert numpy types -> native Python
    if isinstance(results.get("confusion_matrix"), np.ndarray):
        results["confusion_matrix"] = results["confusion_matrix"].tolist()

    save_path = f"{output_dir}/phase_{i}_results.json"

    try:
        with open(save_path, "w") as f:
            json.dump(results, f, indent=4)
        print(f"Saved: {save_path}")
    except Exception as e:
        print(f"Error saving phase {i}:", e)


 Running Grid Search (LightGBM, F1-macro)...

→ Testing params: {'max_depth': 10, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 31}
   F1-macro = 0.7192
→ Testing params: {'max_depth': 10, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 63}
   F1-macro = 0.7445
→ Testing params: {'max_depth': 10, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 127}
   F1-macro = 0.7696

 Best GridSearch Result:
Best Params: {'max_depth': 10, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 127}
Best F1-macro: 0.7696

=== Evaluate Test Set: /kaggle/input/mooccubex-data-cleaned/split_data/data_median_sasmote/test_1.csv ===

Accuracy: 0.9961325515265451
Macro F1: 0.46038859719434005

Confusion Matrix:
[[    19     31    173]
 [     0    106    499]
 [     0    196 231429]]

Per-class Metrics:
{'0': {'precision': 1.0, 'recall': 0.08520179372197309, 'f1-score': 0.15702479338842973, 'support': 223, 'gmean': 0.2918934629654681}, '1': {'precision': 0.3183183183

## 2.3 Thực nghiệm LightGNB trên tập data_median_radiussmote

In [6]:
import json
import os
trainer = LGBMTrainer("/kaggle/input/mooccubex-data-cleaned/split_data/data_median_radiussmote/train_median_radiussmote.csv")
trainer.load_and_prepare_train()

param_grid = {
    "n_estimators": [200],
    "max_depth": [8, 10],
    "num_leaves": [31, 63],
    "min_child_samples": [20]
}


best_model, best_params, best_f1 = trainer.train_with_gridsearch(
    val_path="/kaggle/input/mooccubex-data-cleaned/split_data/data_median_radiussmote/val.csv",
    param_grid=param_grid
)

output_dir = "/kaggle/working/median_radiussmote"
os.makedirs(output_dir, exist_ok=True)   # tạo thư mục nếu chưa có

# Evaluate nhiều test set
for i in range(1, 5):
    results = trainer.evaluate_test(
        f"/kaggle/input/mooccubex-data-cleaned/split_data/data_median_im_3/test_{i}.csv",
        metrics=ClassificationMetrics()
    )

    # Convert numpy types -> native Python
    if isinstance(results.get("confusion_matrix"), np.ndarray):
        results["confusion_matrix"] = results["confusion_matrix"].tolist()

    save_path = f"{output_dir}/phase_{i}_results.json"

    try:
        with open(save_path, "w") as f:
            json.dump(results, f, indent=4)
        print(f"Saved: {save_path}")
    except Exception as e:
        print(f"Error saving phase {i}:", e)


 Running Grid Search (LightGBM, F1-macro)...

→ Testing params: {'max_depth': 8, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 31}
   F1-macro = 0.0021
→ Testing params: {'max_depth': 8, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 63}
   F1-macro = 0.0017
→ Testing params: {'max_depth': 10, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 31}
   F1-macro = 0.0378
→ Testing params: {'max_depth': 10, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 63}
   F1-macro = 0.0017

 Best GridSearch Result:
Best Params: {'max_depth': 10, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 31}
Best F1-macro: 0.0378

=== Evaluate Test Set: /kaggle/input/mooccubex-data-cleaned/split_data/data_median_im_3/test_1.csv ===

Accuracy: 0.0239274175854904
Macro F1: 0.01647531350780709

Confusion Matrix:
[[   222      1      0]
 [   543     62      0]
 [173705  52642   5278]]

Per-class Metrics:
{'0': {'precision': 0.0012724250587493552, 'recall':

## 2.4 Thực nghiệm trên tập data_median_cdsmote

In [7]:
trainer = LGBMTrainer("/kaggle/input/mooccubex-data-cleaned/split_data/data_median_cdsmote/train_median_cdsmote.csv")
trainer.load_and_prepare_train()

param_grid = {
    "n_estimators": [200],
    "max_depth": [8, 10],
    "num_leaves": [31, 63],
    "min_child_samples": [20]
}


best_model, best_params, best_f1 = trainer.train_with_gridsearch(
    val_path="/kaggle/input/mooccubex-data-cleaned/split_data/data_median_cdsmote/val.csv",
    param_grid=param_grid
)

output_dir = "/kaggle/working/data_median_cdsmote"
os.makedirs(output_dir, exist_ok=True)   # tạo thư mục nếu chưa có

# Evaluate nhiều test set
for i in range(1, 5):
    results = trainer.evaluate_test(
        f"/kaggle/input/mooccubex-data-cleaned/split_data/data_median_cdsmote/test_{i}.csv",
        metrics=ClassificationMetrics()
    )

    # Convert numpy types -> native Python
    if isinstance(results.get("confusion_matrix"), np.ndarray):
        results["confusion_matrix"] = results["confusion_matrix"].tolist()

    save_path = f"{output_dir}/phase_{i}_results.json"

    try:
        with open(save_path, "w") as f:
            json.dump(results, f, indent=4)
        print(f"Saved: {save_path}")
    except Exception as e:
        print(f"Error saving phase {i}:", e)


 Running Grid Search (LightGBM, F1-macro)...

→ Testing params: {'max_depth': 8, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 31}
   F1-macro = 0.3706
→ Testing params: {'max_depth': 8, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 63}
   F1-macro = 0.3935
→ Testing params: {'max_depth': 10, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 31}
   F1-macro = 0.3897
→ Testing params: {'max_depth': 10, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 63}
   F1-macro = 0.4175

 Best GridSearch Result:
Best Params: {'max_depth': 10, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 63}
Best F1-macro: 0.4175

=== Evaluate Test Set: /kaggle/input/mooccubex-data-cleaned/split_data/data_median_cdsmote/test_1.csv ===

Accuracy: 0.9891461929938525
Macro F1: 0.35272706864600084

Confusion Matrix:
[[     0     27    196]
 [     0     79    526]
 [     0   1774 229851]]

Per-class Metrics:
{'0': {'precision': 0.0, 'recall': 0.0, 'f1-score

## 2.5 Thực nghiệm trên tập data_extra_trees_cdsmote

In [8]:
import json
import os
trainer = LGBMTrainer("/kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_cdsmote/train_extratrees_cdsmote.csv")
trainer.load_and_prepare_train()

param_grid = {
    "n_estimators": [200],
    "max_depth": [8, 10],
    "num_leaves": [31, 63],
    "min_child_samples": [20]
}

best_model, best_params, best_f1 = trainer.train_with_gridsearch(
    val_path="/kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_cdsmote/val.csv",
    param_grid=param_grid
)

output_dir = "/kaggle/working/extra_trees_cdsmote"
os.makedirs(output_dir, exist_ok=True)   # tạo thư mục nếu chưa có

# Evaluate nhiều test set
for i in range(1, 5):
    results = trainer.evaluate_test(
        f"/kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_cdsmote/test_{i}.csv",
        metrics=ClassificationMetrics()
    )

    # Convert numpy types -> native Python
    if isinstance(results.get("confusion_matrix"), np.ndarray):
        results["confusion_matrix"] = results["confusion_matrix"].tolist()

    save_path = f"{output_dir}/phase_{i}_results.json"

    try:
        with open(save_path, "w") as f:
            json.dump(results, f, indent=4)
        print(f"Saved: {save_path}")
    except Exception as e:
        print(f"Error saving phase {i}:", e)


 Running Grid Search (LightGBM, F1-macro)...

→ Testing params: {'max_depth': 8, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 31}
   F1-macro = 0.7227
→ Testing params: {'max_depth': 8, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 63}
   F1-macro = 0.7444
→ Testing params: {'max_depth': 10, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 31}
   F1-macro = 0.7250
→ Testing params: {'max_depth': 10, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 63}
   F1-macro = 0.7598

 Best GridSearch Result:
Best Params: {'max_depth': 10, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 63}
Best F1-macro: 0.7598

=== Evaluate Test Set: /kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_cdsmote/test_1.csv ===

Accuracy: 0.9963863662761935
Macro F1: 0.33272997088057876

Confusion Matrix:
[[     0      0    223]
 [     0      0    605]
 [     0     12 231613]]

Per-class Metrics:
{'0': {'precision': 0.0, 'recall': 0.0, 'f1-

## 2.6 Thực nghiệm trên tập data_extra_trees_sasmote

In [9]:
import json
import os
trainer = LGBMTrainer("/kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_sasmote/train_extratrees_sasmote.csv")
trainer.load_and_prepare_train()

param_grid = {
    "n_estimators": [200],
    "max_depth": [8, 10],
    "num_leaves": [31, 63],
    "min_child_samples": [20]
}

best_model, best_params, best_f1 = trainer.train_with_gridsearch(
    val_path="/kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_sasmote/val.csv",
    param_grid=param_grid
)

output_dir = "/kaggle/working/data_extra_trees_sassmote"
os.makedirs(output_dir, exist_ok=True)   # tạo thư mục nếu chưa có

# Evaluate nhiều test set
for i in range(1, 5):
    results = trainer.evaluate_test(
        f"/kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_sasmote/test_{i}.csv",
        metrics=ClassificationMetrics()
    )

    # Convert numpy types -> native Python
    if isinstance(results.get("confusion_matrix"), np.ndarray):
        results["confusion_matrix"] = results["confusion_matrix"].tolist()

    save_path = f"{output_dir}/phase_{i}_results.json"

    try:
        with open(save_path, "w") as f:
            json.dump(results, f, indent=4)
        print(f"Saved: {save_path}")
    except Exception as e:
        print(f"Error saving phase {i}:", e)


 Running Grid Search (LightGBM, F1-macro)...

→ Testing params: {'max_depth': 8, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 31}
   F1-macro = 0.7272
→ Testing params: {'max_depth': 8, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 63}
   F1-macro = 0.7508
→ Testing params: {'max_depth': 10, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 31}
   F1-macro = 0.7119
→ Testing params: {'max_depth': 10, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 63}
   F1-macro = 0.7596

 Best GridSearch Result:
Best Params: {'max_depth': 10, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 63}
Best F1-macro: 0.7596

=== Evaluate Test Set: /kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_sasmote/test_1.csv ===

Accuracy: 0.9965455382378373
Macro F1: 0.35976266133768936

Confusion Matrix:
[[     0     10    213]
 [     0     26    579]
 [     0      1 231624]]

Per-class Metrics:
{'0': {'precision': 0.0, 'recall': 0.0, 'f1-

## 2.7 Thực nghiệm trên tập data_extra_trees_radiussmote

In [10]:
import json
import os
trainer = LGBMTrainer("/kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_radiussmote/train_extratrees_radiussmote.csv")
trainer.load_and_prepare_train()
best_model, best_params, best_f1 = trainer.train_with_gridsearch(
    val_path="/kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_radiussmote/val.csv",
    param_grid=param_grid
)

output_dir = "/kaggle/working/data_extra_trees_radiussmote"
os.makedirs(output_dir, exist_ok=True)   # tạo thư mục nếu chưa có

# Evaluate nhiều test set
for i in range(1, 5):
    results = trainer.evaluate_test(
        f"//kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_radiussmote/test_{i}.csv",
        metrics=ClassificationMetrics()
    )

    # Convert numpy types -> native Python
    if isinstance(results.get("confusion_matrix"), np.ndarray):
        results["confusion_matrix"] = results["confusion_matrix"].tolist()

    save_path = f"{output_dir}/phase_{i}_results.json"

    try:
        with open(save_path, "w") as f:
            json.dump(results, f, indent=4)
        print(f"Saved: {save_path}")
    except Exception as e:
        print(f"Error saving phase {i}:", e)


 Running Grid Search (LightGBM, F1-macro)...

→ Testing params: {'max_depth': 8, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 31}
   F1-macro = 0.0332
→ Testing params: {'max_depth': 8, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 63}
   F1-macro = 0.0218
→ Testing params: {'max_depth': 10, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 31}
   F1-macro = 0.0018
→ Testing params: {'max_depth': 10, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 63}
   F1-macro = 0.0017

 Best GridSearch Result:
Best Params: {'max_depth': 8, 'min_child_samples': 20, 'n_estimators': 200, 'num_leaves': 31}
Best F1-macro: 0.0332

=== Evaluate Test Set: //kaggle/input/mooccubex-data-cleaned/split_data/data_extra_trees_radiussmote/test_1.csv ===

Accuracy: 0.0350350393412862
Macro F1: 0.022803335653393268

Confusion Matrix:
[[     0    223      0]
 [     0    605      0]
 [     0 224086   7539]]

Per-class Metrics:
{'0': {'precision': 0.0, 'recall': 0.0,